In [14]:
import aiohttp
import ast
import asyncio
import openai
import pandas as pd
import re
import json

from openai import AsyncClient
from datasets import load_dataset
from tqdm.asyncio import tqdm

In [15]:
async_client = AsyncClient()

In [ ]:
dataset = load_dataset("DataForGood/climateguard")
df = pd.concat([dataset["train"].to_pandas(), dataset["test"].to_pandas()])
df_test = dataset["test"].to_pandas().reset_index()

In [30]:
async def get_labels_async(label_list, transcript, semaphore=None):
    if semaphore:
        await semaphore.acquire()
    try:
        chat_response = await async_client.responses.create(
            model="gpt-5-nano",
            reasoning={"effort": "minimal"},
            #temperature=0,
            input=[{"role": "system", "content": "You are an assistant helping editors to aggregate tv transcripts regarding the climate into themes."},
                {
                    "role": "user",
                    "content": f"""
                            Given a transcript return one or two phrases that best describe the theme of the transcript, avoid bothsides-ism. 
                            You will be given the following example themes as a style guide.
                            Example labels: {json.dumps(label_list, indent=2)}
                            Transcript: {transcript}
                            You should NOT return meaningless label names such as ’new_label_1’ or ’unknown_topic_1’ and only return the new label names, please return the labels as a python list  
                            """
                },
            ]
        )
    finally:
        if semaphore:
            semaphore.release()
    try:
        return ast.literal_eval(chat_response.output_text)
    except SyntaxError:
        return str(chat_response.output_text)

In [86]:
classify_prompts = {
    "0.0": """
        The following themes have been extracted from a tv or radio transcript. Use them as a guide to determine whether the
        transcript contains misinformation on climate change.
        Only classify as misinformation if
        - There is explicit denial of climate science consensus.
        - There is intentional misinformation on policies regarding climate change.
        - Conspiracy theories are being proposed.
        Do not classify as misinformation:
        - Criticism of political or economic decisions on how to tackle climate change.
        - Any affirmations that are not factually verifiable.
        For example saying that proven measures against climate change do not work is misinformation,
        but criticising their implementation or implying that they might have an effect on specific social classes
        (such as fishermen or farmers) is not.
        Transcript: {transcript}
        Themes Extracted: {themes}
        You should not reply with titles or explainations or even text formatting. Reply Yes or No.
        """,
    "0.1": """
        The following themes have been extracted from a tv or radio transcript. Use them as a guide to determine whether the
        transcript contains misinformation on climate change.
        Only classify as misinformation if
        - There is explicit denial of climate science consensus.
        - There is intentional misinformation on policies regarding climate change.
        - Conspiracy theories are being proposed.
        Do not classify as misinformation:
        - Criticism of political or economic decisions on how to tackle climate change.
        - Any affirmations that are not factually verifiable.
        For example saying that proven measures against climate change do not work is misinformation,
        but criticising their implementation or implying that they might have an effect on specific social classes
        (such as fishermen or farmers) is not.
        When in doubt classify as misinformation.
        Transcript: {transcript}
        Themes Extracted: {themes}
        You should not reply with titles or explainations or even text formatting. Reply Yes or No.
        """,
}

In [97]:
async def classify_async(transcript, themes, semaphore=None):
    if semaphore:
        await semaphore.acquire()
    try:
        chat_response = await async_client.responses.create(
            model="gpt-5-nano",
            reasoning={"effort": "minimal"},
            #temperature=0,
            input=[{"role": "system", "content": "You are an assistant helping editors to classify tv transcripts into misinformation or not."},
                {
                    "role": "user",
                    "content": classify_prompts["0.0"].format(transcript=transcript, themes=themes)
                },
            ]
        )
    finally:
        if semaphore:
            semaphore.release()
    return chat_response.output_text


In [139]:
def parse_response(response: str):
    """Parse response containing only a score."""
    match = re.match(r"^[^\d]*(\d+)", response)
    if match:
        score = int(match.group(1))  # Extract score as an integer
    else:
        score = 0
    return score


async def classify_old_async(transcript, idx=None, semaphore=None):
    if semaphore:
        await semaphore.acquire()
    try:
        chat_response = await async_client.responses.create(
            model="gpt-5-nano",
            reasoning={"effort": "minimal"},
            # temperature=0,
            input=[{"role": "system", "content": "You are an assistant helping editors to classify tv transcripts into misinformation or not."},
                {
                    "role": "user",
                    "content": f"""You are an assistant helping editors to moderate TV and radio content.
You will be provided with a transcript delimited by triple backticks.
Bear in mind that the transcript may be missing punctuation and may be of very low quality, with incorrect vocabulary, cuts in the wrong places, or may include some phonetic transcription.

Does the text promote climate change misinformation that undermines well-established scientific consensus, such as denying the existence of climate change or the factors that contribute to it ?

Answer just a number on a scale of 0 - does not promote - to 10 - certainly promotes.

text: {transcript}"""
                },
            ]
        )
    finally:
        if semaphore:
            semaphore.release()
    
    return idx, chat_response.output_text.strip()


In [98]:
label_list = [
    "Les renouvelables augmentent le coût de l'électricité",
    "Il est inutile de réduire les rejets de gaz à effet de serre de la France",
    "Les populations sont manipulées de façon injustifiée.",
    "Il est fondamentale de réduire la quantité de CO2 dans l'atmosphère.",
    "Le rapport du GIEC exprime le consensus scientifique sur le changement climatique",
]

In [99]:
themes = await get_labels_async(label_list, df.iloc[0].plaintext, semaphore=None)
themes

["Le raisonnement politique passe avant les faits sur l'énergie et l’environnement",
 "Les postures idéologiques entravent des solutions pratiques pour l'électricité et la transition"]

In [100]:
await classify_async(df.iloc[0].plaintext, themes)

'No'

In [117]:
await classify_old_async(df.iloc[0].plaintext)

(None, False)

In [114]:
def map_classification(classification):
    if classification.lower() == "yes":
        return True
    elif classification.lower() == "no":
        return False
    elif "no" in classification.lower():
        return False
    elif "yes" in classification.lower():
        return True
    return False

async def get_classification_async(label_list, transcript, idx=None, semaphore=None):
    themes = await get_labels_async(label_list, transcript, semaphore=semaphore)
    classification = await classify_async(transcript, themes)
    return idx, map_classification(classification)



In [115]:
await get_classification_async(label_list, df.iloc[0].plaintext, semaphore=None)

(None, False)

In [ ]:
from tqdm.asyncio import tqdm

semaphore = asyncio.Semaphore(20)
tasks = [
    get_classification_async(
        label_list,
        row["plaintext"],
        idx=idx,
        semaphore=semaphore,
    ) for idx, row in df_test.iterrows()
]
predictions = await tqdm.gather(*tasks)

tasks = [
    classify_old_async(
        row["plaintext"],
        idx=idx,
        semaphore=semaphore,
    ) for idx, row in df_test.iterrows()
]
predictions_old = await tqdm.gather(*tasks)

  0%|          | 0/491 [00:00<?, ?it/s]

100%|██████████| 491/491 [00:25<00:00, 19.38it/s]


In [134]:
predictions = sorted(predictions, key=lambda x: x[0])
predictions_old = sorted(predictions_old, key=lambda x: x[0])

In [145]:
from sklearn.metrics import classification_report

df_test["predictions"] = [element[1] for element in predictions]
df_test["predictions_old"] = [parse_response(element[1])>=5 for element in predictions_old]
print(
    classification_report(
        df_test.misinformation,
        df_test.predictions
    )
)
print(
    classification_report(
        df_test.misinformation,
        df_test.predictions_old
    )
)

              precision    recall  f1-score   support

       False       0.74      0.59      0.66       319
        True       0.44      0.60      0.51       172

    accuracy                           0.60       491
   macro avg       0.59      0.60      0.58       491
weighted avg       0.63      0.60      0.61       491

              precision    recall  f1-score   support

       False       0.67      0.87      0.76       319
        True       0.45      0.19      0.27       172

    accuracy                           0.64       491
   macro avg       0.56      0.53      0.51       491
weighted avg       0.59      0.64      0.59       491



In [144]:
predictions_old

[(0, '3'),
 (1, '3'),
 (2, '1'),
 (3, '0'),
 (4, '4'),
 (5, '0'),
 (6, '0'),
 (7, '5'),
 (8, '4'),
 (9, '6'),
 (10, '0'),
 (11, '3'),
 (12, '0'),
 (13, '0'),
 (14, '3'),
 (15, '4'),
 (16, '3'),
 (17, '3'),
 (18, '4'),
 (19, '0'),
 (20, '4'),
 (21, '0'),
 (22, '0'),
 (23, '2'),
 (24, '3'),
 (25, '0'),
 (26, '2'),
 (27, '6'),
 (28, '0'),
 (29, '0'),
 (30, '0'),
 (31, '0'),
 (32, '6'),
 (33, '0'),
 (34, '3'),
 (35, '2'),
 (36, '3'),
 (37, '4'),
 (38, '0'),
 (39, '3'),
 (40, '5'),
 (41, '2'),
 (42, '0'),
 (43, '2'),
 (44, '0'),
 (45, '6'),
 (46, '6'),
 (47, '0'),
 (48, '0'),
 (49, '3'),
 (50, '0'),
 (51, '2'),
 (52, '3'),
 (53, '0'),
 (54, '7'),
 (55, '4'),
 (56, '0'),
 (57, '0'),
 (58, '4'),
 (59, '0'),
 (60, '3'),
 (61, '0'),
 (62, '4'),
 (63, '4'),
 (64, '4'),
 (65, '4'),
 (66, '3'),
 (67, '2'),
 (68, '2'),
 (69, '4'),
 (70, '0'),
 (71, '0'),
 (72, '0'),
 (73, '2'),
 (74, '5'),
 (75, '0'),
 (76, '1'),
 (77, '0'),
 (78, '3'),
 (79, '0'),
 (80, '3'),
 (81, '0'),
 (82, '2'),
 (83, '0'),
 (